# Actividad 15v5 - GM v3 — NLP corpus ampliado (600 noticias)
**Autor:** Fabrizio Sanchez Saravia - UPeU Juliaca

Identico a GM v2 (actividad_15v2) pero entrenado con el corpus NLP v2
(`sentimiento_mensual_v2.csv`: Agraria.pe + Andina + RedAgricola +
Agronoticias + FreshFruit). Referencia a superar: **GM v2 MAE=0.0646**
(corpus original, 528 noticias).

| Mejora | Descripcion |
|--------|-------------|
| M1 | nlp_index = avg_sentiment x log(n_noticias+1) |
| M2 | nlp_index_lag1 - lag 1 mes |
| M3 | Dropout=0.5 en rama NLP |
| M4 | PCA 95% varianza |
| M5 | Corpus NLP ampliado v2 (multi-fuente) |


In [ ]:
import os, json, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, regularizers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    tf.config.experimental.set_memory_growth(gpus[0], True)
    print(f'GPU: {gpus[0].name}')
else:
    print('CPU mode')
print(f'TF: {tf.__version__}')
PROJECT_ROOT = Path('../..')
DATA_PATH   = PROJECT_ROOT / 'data/processed/master_dataset_fase2_multivariado.csv'
NLP_PATH    = PROJECT_ROOT / 'notebooks/fase2/output/01_nlp_sentimiento/sentimiento_mensual_v2.csv'
GE_METRICAS = PROJECT_ROOT / 'resultados/ge/ge_metricas.json'
GE_PRED     = PROJECT_ROOT / 'resultados/ge/ge_predicciones.csv'
OUT_DIR     = PROJECT_ROOT / 'resultados/gm_v3'
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'DATA_PATH ok: {DATA_PATH.exists()}')
print(f'NLP_PATH  ok: {NLP_PATH.exists()}')

In [ ]:
df_raw = pd.read_csv(DATA_PATH, parse_dates=['fecha_evento'])
print(f'Raw: {df_raw.shape}')
df_master = df_raw.groupby('fecha_evento').mean(numeric_only=True).reset_index()
df_master = df_master.sort_values('fecha_evento').reset_index(drop=True)
print(f'Agregado: {df_master.shape}')
print(f'Rango: {df_master["fecha_evento"].min().date()} -> {df_master["fecha_evento"].max().date()}')

In [ ]:
df_nlp = pd.read_csv(NLP_PATH, encoding='utf-8-sig')
print(f'Columnas NLP: {df_nlp.columns.tolist()}')
fc = [c for c in df_nlp.columns if any(k in c.lower() for k in ['fecha','periodo','mes','month'])][0]
df_nlp = df_nlp.rename(columns={fc: 'fecha_evento'})
df_nlp['fecha_evento'] = pd.to_datetime(df_nlp['fecha_evento'])
df_nlp = df_nlp.sort_values('fecha_evento').reset_index(drop=True)
print(f'NLP: {df_nlp.shape}')
print(df_nlp['avg_sentiment'].describe())
print(df_nlp['n_noticias_beto'].describe())

In [ ]:
df_nlp['nlp_index']      = df_nlp['avg_sentiment'] * np.log1p(df_nlp['n_noticias_beto'])
df_nlp['nlp_index_lag1'] = df_nlp['nlp_index'].shift(1).fillna(0)
print(df_nlp[['fecha_evento','avg_sentiment','n_noticias_beto','nlp_index','nlp_index_lag1']].head(10).to_string())
fig, axes = plt.subplots(2, 2, figsize=(14, 7))
axes[0,0].plot(df_nlp['fecha_evento'], df_nlp['avg_sentiment'], color='steelblue', lw=1.5)
axes[0,0].axhline(0, color='red', ls='--', alpha=0.5)
axes[0,0].set_title('avg_sentiment crudo')
axes[0,0].grid(alpha=0.3)
axes[0,1].bar(df_nlp['fecha_evento'], df_nlp['n_noticias_beto'], color='orange', alpha=0.7)
axes[0,1].set_title('n_noticias_beto crudo')
axes[0,1].grid(alpha=0.3)
axes[1,0].plot(df_nlp['fecha_evento'], df_nlp['nlp_index'], color='darkgreen', lw=1.5)
axes[1,0].axhline(0, color='red', ls='--', alpha=0.5)
axes[1,0].set_title('nlp_index M1')
axes[1,0].grid(alpha=0.3)
axes[1,1].plot(df_nlp['fecha_evento'], df_nlp['nlp_index'], color='darkgreen', lw=1.5, label='t')
axes[1,1].plot(df_nlp['fecha_evento'], df_nlp['nlp_index_lag1'], color='purple', lw=1.5, ls='--', label='lag1')
axes[1,1].legend()
axes[1,1].set_title('nlp_index vs lag-1 M2')
axes[1,1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / 'nlp_features_engineering.png', dpi=150, bbox_inches='tight')
plt.close()
print('Grafico NLP guardado')

In [ ]:
df = df_master.merge(df_nlp[['fecha_evento','nlp_index','nlp_index_lag1']], on='fecha_evento', how='left')
df['nlp_index']      = df['nlp_index'].fillna(0)
df['nlp_index_lag1'] = df['nlp_index_lag1'].fillna(0)
df = df.sort_values('fecha_evento').reset_index(drop=True)
TARGET = 'produccion_t'
META   = ['fecha_evento', TARGET]
STRUCT = [c for c in df_master.columns if c not in META]
NLP_F  = ['nlp_index', 'nlp_index_lag1']
print(f'Dataset fusionado: {df.shape}')
print(f'Struct ({len(STRUCT)}): {STRUCT}')
print(f'NLP: {NLP_F}')

In [ ]:
TIMESTEPS = 6
n_total = len(df)
n_train = int(n_total * 0.80)
n_test  = n_total - n_train
df_train = df.iloc[:n_train].copy()
df_test  = df.iloc[n_train:].copy()
print(f'Train: {n_train} | {df_train["fecha_evento"].min().date()} -> {df_train["fecha_evento"].max().date()}')
print(f'Test:  {n_test}  | {df_test["fecha_evento"].min().date()} -> {df_test["fecha_evento"].max().date()}')
scaler_s = StandardScaler()
scaler_n = StandardScaler()
scaler_y = StandardScaler()
Xs_tr_raw = scaler_s.fit_transform(df_train[STRUCT])
Xs_te_raw = scaler_s.transform(df_test[STRUCT])
Xn_tr_raw = scaler_n.fit_transform(df_train[NLP_F])
Xn_te_raw = scaler_n.transform(df_test[NLP_F])
y_tr_sc   = scaler_y.fit_transform(df_train[[TARGET]])
y_te_sc   = scaler_y.transform(df_test[[TARGET]])
print('Escalado OK')

In [ ]:
pca = PCA(n_components=0.95, random_state=SEED)
Xs_tr = pca.fit_transform(Xs_tr_raw)
Xs_te = pca.transform(Xs_te_raw)
n_comp   = pca.n_components_
var_acum = np.cumsum(pca.explained_variance_ratio_)
print(f'PCA: {Xs_tr_raw.shape[1]} -> {n_comp} componentes')
for i,(v,a) in enumerate(zip(pca.explained_variance_ratio_, var_acum)):
    print(f'  PC{i+1}: {v:.3f} acum={a:.3f}')
fig, ax = plt.subplots(figsize=(8,4))
ax.bar(range(1,n_comp+1), pca.explained_variance_ratio_, color='steelblue', alpha=0.7)
ax2 = ax.twinx()
ax2.plot(range(1,n_comp+1), var_acum, 'ro-', lw=2)
ax2.axhline(0.95, color='red', ls='--', alpha=0.5)
ax.set_title(f'PCA M4: {Xs_tr_raw.shape[1]} -> {n_comp} componentes')
plt.tight_layout()
plt.savefig(OUT_DIR / 'pca_varianza_explicada.png', dpi=150, bbox_inches='tight')
plt.close()
print('Grafico PCA guardado')

In [ ]:
def make_seq(Xs, Xn, y, ts):
    a, b, c = [], [], []
    for i in range(ts, len(Xs)):
        a.append(Xs[i-ts:i])
        b.append(Xn[i-ts:i])
        c.append(y[i])
    return np.array(a), np.array(b), np.array(c)

Xs_seq_tr, Xn_seq_tr, y_seq_tr = make_seq(Xs_tr, Xn_tr_raw, y_tr_sc, TIMESTEPS)
Xs_seq_te, Xn_seq_te, y_seq_te = make_seq(Xs_te, Xn_te_raw, y_te_sc, TIMESTEPS)
print(f'Xs_train: {Xs_seq_tr.shape}')
print(f'Xn_train: {Xn_seq_tr.shape}')
print(f'Xs_test:  {Xs_seq_te.shape}')
print(f'Xn_test:  {Xn_seq_te.shape}')

In [ ]:
def build_gm_v3(ss, ns, units=64, drs=0.2, drn=0.5):
    inp_s = layers.Input(shape=ss, name='inp_struct')
    h  = layers.LSTM(units, return_sequences=True)(inp_s)
    sc = layers.Dense(1, activation='tanh')(h)
    sw = layers.Softmax(axis=1)(sc)
    ca = layers.Multiply()([h, sw])
    ca = layers.Lambda(lambda x: tf.reduce_sum(x, axis=1))(ca)
    ca = layers.Dropout(drs)(ca)
    inp_n = layers.Input(shape=ns, name='inp_nlp')
    cb = layers.LSTM(16, return_sequences=False)(inp_n)
    cb = layers.Dropout(drn, name='dropout_nlp_M3')(cb)
    mg = layers.Concatenate()([ca, cb])
    x  = layers.Dense(32, activation='relu', kernel_regularizer=regularizers.l2(1e-4))(mg)
    x  = layers.Dropout(0.2)(x)
    x  = layers.Dense(16, activation='relu')(x)
    out= layers.Dense(1)(x)
    return Model(inputs=[inp_s, inp_n], outputs=out, name='GM_v3')

ss = (Xs_seq_tr.shape[1], Xs_seq_tr.shape[2])
ns = (Xn_seq_tr.shape[1], Xn_seq_tr.shape[2])
model = build_gm_v3(ss, ns)
model.summary()
print(f'struct={ss} nlp={ns} | M3: Dropout NLP=0.5')

In [ ]:
model.compile(optimizer=keras.optimizers.Adam(1e-3), loss='mse', metrics=['mae'])
callbacks = [
    EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=7, min_lr=1e-6, verbose=1)
]
print('Entrenando GM v3...')
history = model.fit(
    [Xs_seq_tr, Xn_seq_tr], y_seq_tr,
    epochs=200, batch_size=8, validation_split=0.2,
    callbacks=callbacks, shuffle=False, verbose=1
)

In [ ]:
best_epoch    = int(np.argmin(history.history['val_loss'])) + 1
best_val_loss = float(min(history.history['val_loss']))
fig, axes = plt.subplots(1, 2, figsize=(12,4))
axes[0].plot(history.history['loss'],     label='Train', color='steelblue')
axes[0].plot(history.history['val_loss'], label='Val',   color='orange')
axes[0].set_title('Loss MSE')
axes[0].legend()
axes[0].grid(alpha=0.3)
axes[1].plot(history.history['mae'],     label='Train', color='steelblue')
axes[1].plot(history.history['val_mae'], label='Val',   color='orange')
axes[1].set_title('MAE')
axes[1].legend()
axes[1].grid(alpha=0.3)
plt.suptitle(f'GM v3 | best_epoch={best_epoch} | val_loss={best_val_loss:.4f}', fontweight='bold')
plt.tight_layout()
plt.savefig(OUT_DIR / 'gm_v3_training_curves.png', dpi=150, bbox_inches='tight')
plt.close()
print(f'Best epoch={best_epoch} val_loss={best_val_loss:.4f}')

In [ ]:
y_pred_sc = model.predict([Xs_seq_te, Xn_seq_te], verbose=0)
y_pred = scaler_y.inverse_transform(y_pred_sc).flatten()
y_true = scaler_y.inverse_transform(y_seq_te).flatten()
mae  = float(mean_absolute_error(y_true, y_pred))
rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
r2   = float(r2_score(y_true, y_pred))
mape = float(np.mean(np.abs((y_true - y_pred) / (np.abs(y_true) + 1e-8))) * 100)
if GE_METRICAS.exists():
    with open(GE_METRICAS) as f:
        ge = json.load(f)
    ge_mae  = ge.get('mae',  ge.get('MAE',  0.0673))
    ge_rmse = ge.get('rmse', ge.get('RMSE', 0.0698))
    ge_r2   = ge.get('r2',   ge.get('R2',   -2.34))
else:
    ge_mae, ge_rmse, ge_r2 = 0.0673, 0.0698, -2.34
gm_orig = {'mae':0.0981,'rmse':0.1007,'r2':-5.96}
gm_v2 = {'mae': 0.0646, 'rmse': 0.0771, 'r2': -9.86}  # referencia: corpus original 528 noticias
print('=' * 60)
print('COMPARATIVA FINAL')
print('=' * 60)
print(f"{'Metrica':<10} {'GE':>13} {'GM orig':>10} {'GM v2':>10} {'GM v3':>10}")
print(f"{'MAE':<10} {ge_mae:>13.4f} {gm_orig['mae']:>10.4f} {gm_v2['mae']:>10.4f} {mae:>10.4f}")
print(f"{'RMSE':<10} {ge_rmse:>13.4f} {gm_orig['rmse']:>10.4f} {gm_v2['rmse']:>10.4f} {rmse:>10.4f}")
print(f"{'R2':<10} {ge_r2:>13.4f} {gm_orig['r2']:>10.4f} {gm_v2['r2']:>10.4f} {r2:>10.4f}")
print(f"{'MAPE%':<10} {'N/A':>13} {'365.0':>10} {'N/A':>10} {mape:>10.1f}")
print('=' * 60)
if mae < ge_mae:
    print(f'SUPERA GE: {(ge_mae-mae)/ge_mae*100:.1f}% mejor')
elif mae < gm_orig['mae']:
    print(f'Mejora sobre GM orig: {(gm_orig["mae"]-mae)/gm_orig["mae"]*100:.1f}%')
    print(f'Aun {(mae-ge_mae)/ge_mae*100:.1f}% peor que GE')
else:
    print('Sin mejora - desalineacion geografica NLP confirmada')
if mae < gm_v2['mae']:
    print(f"SUPERA GM v2 (corpus original): {(gm_v2['mae']-mae)/gm_v2['mae']*100:.1f}% mejor")
else:
    print(f"No supera GM v2 (corpus original): {mae:.4f} vs {gm_v2['mae']:.4f}")


In [ ]:
fechas_test = df['fecha_evento'].iloc[n_train + TIMESTEPS:].reset_index(drop=True)
fig, ax = plt.subplots(figsize=(12,5))
ax.plot(fechas_test, y_true, 'o-',  color='black',     lw=2, ms=5, label='Real')
ax.plot(fechas_test, y_pred, 's--', color='darkorange', lw=2, ms=5, label=f'GM v3 MAE={mae:.4f}')
if GE_PRED.exists():
    df_ge = pd.read_csv(GE_PRED)
    col_pred = [c for c in df_ge.columns if 'pred' in c.lower()]
    if col_pred:
        n_ov = min(len(fechas_test), len(df_ge))
        ax.plot(fechas_test[:n_ov], df_ge[col_pred[0]].values[:n_ov],
                '^:', color='steelblue', lw=1.5, ms=5, alpha=0.7, label='GE 0.0673')
ax.set_title('GM v3 - Predicciones vs Real', fontweight='bold')
ax.set_xlabel('Fecha')
ax.set_ylabel('Produccion (media provincial)')
ax.legend()
ax.grid(alpha=0.3)
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig(OUT_DIR / 'gm_v3_predicciones_vs_real.png', dpi=150, bbox_inches='tight')
plt.close()
print('Grafico predicciones guardado')

In [ ]:
print('Ablation: sin lag M2...')
sc_nl = StandardScaler()
Xn_tr_nl = sc_nl.fit_transform(df_train[['nlp_index']])
Xn_te_nl = sc_nl.transform(df_test[['nlp_index']])
_, Xn_seq_tr_nl, _ = make_seq(Xs_tr, Xn_tr_nl, y_tr_sc, TIMESTEPS)
_, Xn_seq_te_nl, _ = make_seq(Xs_te, Xn_te_nl, y_te_sc, TIMESTEPS)
m_nl = build_gm_v3((Xs_seq_tr.shape[1], Xs_seq_tr.shape[2]),
                   (Xn_seq_tr_nl.shape[1], Xn_seq_tr_nl.shape[2]))
m_nl.compile(optimizer=keras.optimizers.Adam(1e-3), loss='mse', metrics=['mae'])
m_nl.fit([Xs_seq_tr, Xn_seq_tr_nl], y_seq_tr,
         epochs=200, batch_size=8, validation_split=0.2,
         callbacks=[EarlyStopping(monitor='val_loss', patience=15,
                                  restore_best_weights=True, verbose=0)],
         shuffle=False, verbose=0)
y_pred_nl = scaler_y.inverse_transform(
    m_nl.predict([Xs_seq_te, Xn_seq_te_nl], verbose=0)).flatten()
mae_nl  = float(mean_absolute_error(y_true, y_pred_nl))
rmse_nl = float(np.sqrt(mean_squared_error(y_true, y_pred_nl)))
r2_nl   = float(r2_score(y_true, y_pred_nl))
print('ABLATION')
print(f"{'GM original':<32} MAE=0.0981 RMSE=0.1007 R2=-5.96")
print(f"{'GM v3 sin lag M1+M3+M4':<32} MAE={mae_nl:.4f} RMSE={rmse_nl:.4f} R2={r2_nl:.4f}")
print(f"{'GM v3 completo M1+M2+M3+M4':<32} MAE={mae:.4f} RMSE={rmse:.4f} R2={r2:.4f}")
print(f"{'GM v2 corpus original':<32} MAE=0.0646 RMSE=0.0771 R2=-9.86")
print(f"{'GE baseline sin NLP':<32} MAE=0.0673 RMSE=0.0698 R2=-2.34")
delta = mae_nl - mae
print(f'Lag M2: {delta:+.4f} ({"mejoro" if delta>0 else "no aporto"})')

In [ ]:
resultados = {
    'modelo': 'GM_v3_DualLSTM_NLP_CorpusAmpliado',
    'mejoras': ['M1_nlp_index','M2_lag1','M3_dropout_nlp_0.5','M4_pca_95pct'],
    'MAE': mae, 'RMSE': rmse, 'R2': r2, 'MAPE': mape,
    'n_train': n_train, 'n_test': n_test,
    'pca_components': int(n_comp),
    'best_val_loss': best_val_loss,
    'best_epoch': best_epoch,
    'timesteps': TIMESTEPS,
    'comparativa': {
        'GE_sin_NLP':  {'MAE': ge_mae,        'RMSE': ge_rmse,         'R2': ge_r2},
        'GM_original': {'MAE': gm_orig['mae'], 'RMSE': gm_orig['rmse'], 'R2': gm_orig['r2']},
        'GM_v2_corpus_orig': {'MAE': gm_v2['mae'], 'RMSE': gm_v2['rmse'], 'R2': gm_v2['r2']},
        'GM_v3':       {'MAE': mae,            'RMSE': rmse,            'R2': r2}
    },
    'ablation_M2': {
        'sin_lag': {'MAE': mae_nl, 'RMSE': rmse_nl, 'R2': r2_nl},
        'con_lag': {'MAE': mae,    'RMSE': rmse,    'R2': r2}
    }
}
with open(OUT_DIR / 'gm_v3_metricas.json', 'w') as f:
    json.dump(resultados, f, indent=2)
pd.DataFrame({'fecha': fechas_test.values, 'real': y_true, 'pred_gm_v3': y_pred}).to_csv(
    OUT_DIR / 'gm_v3_predicciones.csv', index=False)
model.save(OUT_DIR / 'gm_v3_model.keras')
print('Archivos en resultados/gm_v3/:')
for f in sorted(OUT_DIR.iterdir()):
    print(f'  {f.name}')
print()
print('RESUMEN EJECUTIVO')
print(f'GE:      MAE={ge_mae:.4f} RMSE={ge_rmse:.4f} R2={ge_r2:.4f}')
print(f'GM orig: MAE=0.0981  RMSE=0.1007  R2=-5.96')
print(f'GM v2:   MAE=0.0646  RMSE=0.0771  R2=-9.86')
print(f'GM v3:   MAE={mae:.4f} RMSE={rmse:.4f} R2={r2:.4f}')
if mae < ge_mae:
    print('RESULTADO: NLP mejorado SUPERA al GE')
elif mae < 0.0981:
    print('RESULTADO: Mejora parcial sobre GM original')
else:
    print('RESULTADO: Desalineacion geografica NLP-target confirmada')